In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (average_precision_score, f1_score,
                             balanced_accuracy_score)
from imblearn.pipeline import Pipeline      # imblearn, not sklearn
from imblearn.over_sampling import SMOTE
import numpy as np

# 1. Simple loader function with binarization (minority class = 1)
def load_dataset(dataset_name):
    dataset = fetch_openml(name=dataset_name, version=1, as_frame=True, parser="auto")
    X = dataset.data.select_dtypes(include=[np.number])
    y = dataset.target
    
    # Binarize targets: minority class is 1, rest are 0
    counts = y.value_counts()
    minority_class = counts.idxmin()
    y = (y == minority_class).astype(int)
    return X, y

# Load the haberman dataset as instructed
X, y = load_dataset("haberman")

# 2. Build the leakage-free pipeline
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("resample", SMOTE(random_state=42)),
    ("clf", DecisionTreeClassifier(random_state=42)),
])

# 3. 5-Fold Stratified Cross-Validation loop
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Fold | AUPRC | Macro F1 | Balanced Accuracy")
print("-" * 40)
for fold, (tr, te) in enumerate(cv.split(X, y)):
    pipe.fit(X.iloc[tr], y.iloc[tr])
    proba = pipe.predict_proba(X.iloc[te])[:, 1]
    pred = pipe.predict(X.iloc[te])
    
    print(fold, " | ",
          round(average_precision_score(y.iloc[te], proba), 4), " | ",
          round(f1_score(y.iloc[te], pred, average="macro"), 4), " | ",
          round(balanced_accuracy_score(y.iloc[te], pred), 4))

Fold | AUPRC | Macro F1 | Balanced Accuracy
----------------------------------------
0  |  0.3715  |  0.6209  |  0.617
1  |  0.3301  |  0.6022  |  0.5986
2  |  0.2489  |  0.4438  |  0.4514
3  |  0.3463  |  0.5216  |  0.5403
4  |  0.3105  |  0.5703  |  0.5674


In [2]:
from imblearn.under_sampling import RandomUnderSampler, TomekLinks

# Swap out SMOTE for RandomUnderSampler to check that numbers change
pipe.set_params(resample=RandomUnderSampler(random_state=42))

# Re-run a quick fold test or print a fold
for fold, (tr, te) in enumerate(cv.split(X, y)):
    pipe.fit(X.iloc[tr], y.iloc[tr])
    proba = pipe.predict_proba(X.iloc[te])[:, 1]
    print(f"RUS Fold {fold} AUPRC:", round(average_precision_score(y.iloc[te], proba), 4))
    break  # just test the first fold to verify

RUS Fold 0 AUPRC: 0.2827
